In [ ]:
#pip install --user scikit-learn

In [ ]:
# PACE-VCF Inference and Comparison with MODIS VCF Collection 6 (2020)
## Subset run: 5 tiles only (h09v05, h12v04, h12v09, h20v06, h31v11)
## XGBoost legacy model -- outputs isolated under outputs/xgboost-legacy/
## (does not touch 7f_inference_C6_c61.ipynb's inference/ outputs)

# =============================================================================
# Cell 1: Configuration and Imports
# =============================================================================

import os
os.umask(0o022)

import numpy as np
import pandas as pd
from pathlib import Path
import rasterio
import rasterio.transform
import rasterio.crs
import xgboost as xgb
import subprocess
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_PATH = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/models/pace_vcf_mc_xgb_20260911_153454.json")

PACE_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/output")
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/outputs/xgboost-legacy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_TILES = [
    "h09v05", "h12v04", "h12v09", "h20v06", "h31v11",
]

PACE_YEAR = 2025
MODIS_VCF_YEAR = 2020

TILE_SIZE_MODIS = 4800
TILE_SIZE_PACE = 600
AGGREGATION_FACTOR = 8
NO_DATA = -10001
NO_DATA_OUT = 255

MODIS_UPPER_LEFT_X = -20015109.354
MODIS_UPPER_LEFT_Y = 10007554.677
MODIS_TILE_SIZE_M = 1111950.5196666666

# MODIS VCF Collection 6 (2020)
MODIS_VCF_C6_DIR = Path(f"/css/modis/Collection6/L3/MOD44B-VCF/{MODIS_VCF_YEAR}/065/")

print("="*70)
print("PACE-VCF INFERENCE AND COMPARISON WITH MODIS C6 (2020)")
print("="*70)
print(f"Model: {MODEL_PATH.name}")
print(f"Test tiles: {TEST_TILES}")
print(f"PACE Year: {PACE_YEAR}")
print(f"MODIS VCF C6 Year: {MODIS_VCF_YEAR}")
print(f"MODIS VCF C6 Directory: {MODIS_VCF_C6_DIR}")
print(f"Output: {OUTPUT_DIR}")
print("="*70)

# Check if C6 directory exists and list contents
print(f"\nChecking MODIS C6 directory...")
if MODIS_VCF_C6_DIR.exists():
    files = list(MODIS_VCF_C6_DIR.glob("*.hdf"))
    print(f"  Directory exists: {MODIS_VCF_C6_DIR}")
    print(f"  HDF files found: {len(files)}")
    if files:
        print(f"  Example file: {files[0].name}")
else:
    print(f"  WARNING: Directory not found: {MODIS_VCF_C6_DIR}")


# =============================================================================
# Cell 2: Load Model and Feature List
# =============================================================================

model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

print(f"Model loaded: {MODEL_PATH.name}")
print(f"  Number of features: {model.n_features_in_}")

# NOTE: was previously `MODEL_PATH.stem.split('_')[-1]`, which only grabs
# the time portion of the "..._{date}_{time}" stem (e.g. "164839" instead of
# "20260831_164839"), so this always missed the real companion file and
# silently fell back to whatever mc_top_features_*.txt was newest in the
# directory -- which quietly mismatched the loaded model once other model
# runs (e.g. with different feature sets) landed in the same folder.
timestamp = MODEL_PATH.stem.replace('pace_vcf_mc_xgb_', '')
feature_list_path = MODEL_PATH.parent / f"mc_top_features_{timestamp}.txt"

if not feature_list_path.exists():
    raise FileNotFoundError(
        f"No matching feature list for model {MODEL_PATH.name} -- expected {feature_list_path}"
    )

if feature_list_path.exists():
    with open(feature_list_path, 'r') as f:
        FEATURE_NAMES = [line.strip() for line in f.readlines() if line.strip()]
    print(f"  Features loaded from: {feature_list_path.name}")
    print(f"  Number of features: {len(FEATURE_NAMES)}")
else:
    raise FileNotFoundError(f"No feature list found. Expected: {feature_list_path}")

print(f"\nTop features:")
for i, f in enumerate(FEATURE_NAMES[:10]):
    print(f"  {i+1}. {f}")
if len(FEATURE_NAMES) > 10:
    print(f"  ... and {len(FEATURE_NAMES) - 10} more")


# =============================================================================
# Cell 3: Helper Functions - Geospatial
# =============================================================================

def get_sinusoidal_crs():
    return rasterio.crs.CRS.from_proj4(
        "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs"
    )


def get_transform(tile: str, tile_size: int):
    h = int(tile[1:3])
    v = int(tile[4:6])
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    pixel_size = MODIS_TILE_SIZE_M / tile_size
    return rasterio.transform.from_origin(min_x, max_y, pixel_size, pixel_size)


def write_vcf_geotiff(data: np.ndarray, output_path: Path, tile: str, 
                       description: str = "Percent Tree Cover") -> bool:
    tile_size = data.shape[0]
    out_data = np.where(np.isfinite(data) & (data >= 0) & (data <= 100), 
                        np.round(data).astype(np.uint8), NO_DATA_OUT)
    
    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=tile_size,
        width=tile_size,
        count=1,
        dtype=np.uint8,
        crs=get_sinusoidal_crs(),
        transform=get_transform(tile, tile_size),
        nodata=NO_DATA_OUT,
        compress='lzw',
        tiled=True
    ) as dst:
        dst.write(out_data, 1)
        dst.set_band_description(1, description)
    return True


print("Geospatial helper functions defined ✓")


# =============================================================================
# Cell 4: MODIS VCF C6 Extraction Functions
# =============================================================================

def find_modis_vcf_file_c6(tile: str) -> Path:
    """Find MODIS VCF file in Collection 6 (2020) directory."""
    if not MODIS_VCF_C6_DIR.exists():
        print(f"  WARNING: C6 directory not found: {MODIS_VCF_C6_DIR}")
        return None
    
    # Try different patterns for C6 files
    patterns = [
        f"MOD44B.A2020*.{tile}.006.*.hdf",
        f"MOD44B.*.{tile}.006.*.hdf",
        f"*{tile}*.hdf",
    ]
    
    for pattern in patterns:
        matches = list(MODIS_VCF_C6_DIR.glob(pattern))
        if matches:
            return matches[0]
    
    print(f"  WARNING: No MODIS VCF C6 found for {tile}")
    print(f"    Searched in: {MODIS_VCF_C6_DIR}")
    print(f"    Patterns tried: {patterns}")
    return None


def extract_modis_vcf_via_gdal_translate(hdf_path: Path, output_dir: Path, tile: str) -> np.ndarray:
    """Extract Percent_Tree_Cover from MODIS VCF HDF file."""
    result = subprocess.run(['gdalinfo', str(hdf_path)], capture_output=True, text=True)
    
    subds_path = None
    
    # Look for Percent_Tree_Cover subdataset
    for line in result.stdout.split('\n'):
        if 'SUBDATASET_' in line and 'NAME=' in line and 'Percent_Tree_Cover' in line:
            subds_path = line.split('=', 1)[1].strip()
            break
    
    if subds_path is None:
        # Try alternative search
        for line in result.stdout.split('\n'):
            if 'HDF4_EOS:EOS_GRID' in line and 'Percent_Tree_Cover' in line:
                if '=' in line:
                    subds_path = line.split('=', 1)[1].strip()
                else:
                    subds_path = line.strip()
                break
    
    if subds_path is None:
        print(f"  ERROR: Could not find Percent_Tree_Cover subdataset")
        print(f"  Available subdatasets:")
        for line in result.stdout.split('\n'):
            if 'SUBDATASET_' in line:
                print(f"    {line}")
        return None
    
    temp_tif = output_dir / f"temp_{tile}_pct_tree_c6.tif"
    cmd = ['gdal_translate', '-of', 'GTiff', '-co', 'COMPRESS=LZW', subds_path, str(temp_tif)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"  ERROR: gdal_translate failed: {result.stderr}")
        return None
    
    with rasterio.open(temp_tif) as src:
        tree_cover = src.read(1).astype(np.float32)
    
    temp_tif.unlink()
    tree_cover[(tree_cover > 100) | (tree_cover < 0)] = np.nan
    return tree_cover


def aggregate_to_pace_resolution(data_4800: np.ndarray) -> np.ndarray:
    """Aggregate 4800x4800 to 600x600 resolution."""
    data_reshaped = data_4800.reshape(TILE_SIZE_PACE, AGGREGATION_FACTOR, 
                                       TILE_SIZE_PACE, AGGREGATION_FACTOR)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        data_600 = np.nanmean(data_reshaped, axis=(1, 3))
    return data_600.astype(np.float32)


def process_modis_vcf_tile_c6(tile: str, output_dir: Path) -> dict:
    """Process MODIS VCF Collection 6 (2020) tile."""
    print(f"\n  Processing MODIS VCF C6 (2020) for {tile}...")
    
    hdf_path = find_modis_vcf_file_c6(tile)
    if hdf_path is None:
        return None
    
    print(f"    Source: {hdf_path.name}")
    tree_cover_4800 = extract_modis_vcf_via_gdal_translate(hdf_path, output_dir, tile)
    if tree_cover_4800 is None:
        return None
    
    valid_pct = 100 * np.sum(np.isfinite(tree_cover_4800)) / tree_cover_4800.size
    print(f"    Native resolution: {tree_cover_4800.shape}, {valid_pct:.1f}% valid")
    print(f"    Value range: {np.nanmin(tree_cover_4800):.1f} - {np.nanmax(tree_cover_4800):.1f}%")
    print(f"    Mean: {np.nanmean(tree_cover_4800):.1f}%")
    
    native_path = output_dir / f"MODIS_VCF_C6_2020_{tile}_250m.tif"
    write_vcf_geotiff(tree_cover_4800, native_path, tile, f"MODIS VCF C6 {tile} 2020 250m")
    print(f"    Saved native: {native_path.name}")
    
    tree_cover_600 = aggregate_to_pace_resolution(tree_cover_4800)
    valid_pct_600 = 100 * np.sum(np.isfinite(tree_cover_600)) / tree_cover_600.size
    print(f"    PACE resolution: {tree_cover_600.shape}, {valid_pct_600:.1f}% valid")
    print(f"    Aggregated mean: {np.nanmean(tree_cover_600):.1f}%")
    
    pace_path = output_dir / f"MODIS_VCF_C6_2020_{tile}_2km.tif"
    write_vcf_geotiff(tree_cover_600, pace_path, tile, f"MODIS VCF C6 {tile} 2020 2km")
    print(f"    Saved aggregated: {pace_path.name}")
    
    return {
        'native_path': native_path,
        'pace_path': pace_path,
        'native_data': tree_cover_4800,
        'pace_data': tree_cover_600
    }


print("MODIS VCF C6 extraction functions defined ✓")


# =============================================================================
# Cell 5: PACE Feature Extraction
# =============================================================================

def extract_features_from_metrics(tile: str, feature_names: list) -> tuple:
    """Load features directly from pre-computed metrics files using rasterio."""
    metrics_dir = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics"
    
    n_pixels = TILE_SIZE_PACE * TILE_SIZE_PACE
    n_features = len(feature_names)
    X = np.full((n_pixels, n_features), np.nan, dtype=np.float32)
    
    file_data = {}
    for filename in ['MODIS_Metrics.tif', 'PACE_Metrics.tif', 'PACE_AltSort_Metrics.tif']:
        filepath = metrics_dir / filename
        if filepath.exists():
            with rasterio.open(filepath) as src:
                band_index = {desc: i + 1 for i, desc in enumerate(src.descriptions) if desc}
                file_data[filename] = {'path': filepath, 'band_index': band_index}
        else:
            print(f"    ⚠️ File not found: {filepath}")
    
    found_count = 0
    missing_features = []
    
    for i, feat_name in enumerate(feature_names):
        found = False
        
        for filename, fdata in file_data.items():
            if feat_name in fdata['band_index']:
                with rasterio.open(fdata['path']) as src:
                    data = src.read(fdata['band_index'][feat_name]).astype(np.float32)
                    data = np.where(data == NO_DATA, np.nan, data)
                    data = np.where(data == -255, np.nan, data)
                    X[:, i] = data.flatten()
                found_count += 1
                found = True
                break
        
        if not found:
            missing_features.append(feat_name)
    
    print(f"    Features found: {found_count}/{n_features}")
    if missing_features:
        print(f"    ⚠️ Missing features ({len(missing_features)}): {missing_features[:5]}{'...' if len(missing_features) > 5 else ''}")
    
    nan_per_pixel = np.sum(np.isnan(X), axis=1)
    valid_mask = nan_per_pixel < (n_features * 0.5)
    valid_mask_2d = valid_mask.reshape(TILE_SIZE_PACE, TILE_SIZE_PACE)
    
    valid_pct = 100 * np.sum(valid_mask) / n_pixels
    print(f"    Valid pixels (≥50% features): {np.sum(valid_mask):,} / {n_pixels:,} ({valid_pct:.1f}%)")
    
    total_nan = np.sum(np.isnan(X))
    total_values = X.size
    print(f"    NaN values: {total_nan:,} / {total_values:,} ({100*total_nan/total_values:.1f}%)")
    
    return X, valid_mask_2d


def run_inference(X: np.ndarray, valid_mask_2d: np.ndarray, 
                  model: xgb.XGBRegressor, 
                  modis_mask: np.ndarray = None) -> tuple:
    """Run inference using XGBoost native NaN handling."""
    y_pred = model.predict(X)
    
    vcf_pred = y_pred.reshape(TILE_SIZE_PACE, TILE_SIZE_PACE)
    vcf_pred[~valid_mask_2d] = np.nan
    
    final_mask = valid_mask_2d.copy()
    if modis_mask is not None:
        vcf_pred[~modis_mask] = np.nan
        final_mask = final_mask & modis_mask
    
    vcf_pred = np.clip(vcf_pred, 0, 100)
    return vcf_pred, final_mask


print("PACE feature extraction functions defined ✓")


# =============================================================================
# Cell 6: Process All Test Tiles
# =============================================================================

print("="*70)
print("PROCESSING TEST TILES")
print("="*70)

results = {}

for tile in TEST_TILES:
    print(f"\n{'='*60}")
    print(f"TILE: {tile}")
    print(f"{'='*60}")
    
    metrics_dir = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics"
    if not metrics_dir.exists():
        print(f"  SKIP: PACE metrics not found at {metrics_dir}")
        continue
    
    # Process MODIS VCF C6 (2020)
    modis_result = process_modis_vcf_tile_c6(tile, OUTPUT_DIR)
    
    modis_mask = None
    if modis_result is not None and modis_result['pace_data'] is not None:
        modis_mask = np.isfinite(modis_result['pace_data'])
        print(f"    MODIS C6 valid mask: {np.sum(modis_mask):,} / {modis_mask.size:,} pixels ({100*np.mean(modis_mask):.1f}%)")
    
    print(f"\n  Running PACE VCF inference...")
    print(f"    Extracting features from metrics files...")
    X, valid_mask_2d = extract_features_from_metrics(tile, FEATURE_NAMES)
    
    print(f"    Running model inference...")
    pace_vcf, final_mask = run_inference(X, valid_mask_2d, model, modis_mask=modis_mask)
    
    n_valid = np.sum(final_mask)
    print(f"    Valid pixels (after MODIS mask): {n_valid:,} / {TILE_SIZE_PACE*TILE_SIZE_PACE:,} ({100*n_valid/(TILE_SIZE_PACE*TILE_SIZE_PACE):.1f}%)")
    
    valid_data = pace_vcf[final_mask]
    if len(valid_data) > 0:
        print(f"    Prediction range: {np.nanmin(valid_data):.1f}% - {np.nanmax(valid_data):.1f}%")
        print(f"    Prediction mean: {np.nanmean(valid_data):.1f}%")
    
    pace_vcf_path = OUTPUT_DIR / f"PACE_VCF_{PACE_YEAR}_{tile}_2km.tif"
    write_vcf_geotiff(pace_vcf, pace_vcf_path, tile, f"PACE VCF {tile} {PACE_YEAR}")
    print(f"    Saved: {pace_vcf_path.name}")
    
    results[tile] = {
        'pace_vcf': pace_vcf,
        'pace_vcf_path': pace_vcf_path,
        'valid_mask': final_mask,
        'modis_mask': modis_mask,
    }
    
    if modis_result:
        results[tile].update({
            'modis_vcf_native': modis_result['native_data'],
            'modis_vcf_pace': modis_result['pace_data'],
            'modis_native_path': modis_result['native_path'],
            'modis_pace_path': modis_result['pace_path'],
        })
    
    print(f"\n  ✓ Completed {tile}")

print(f"\n{'='*70}")
print(f"Processing complete: {len(results)} tiles")
print(f"{'='*70}")


# =============================================================================
# Cell 7: Statistical Comparison
# =============================================================================

print("="*70)
print("STATISTICAL COMPARISON: PACE VCF vs MODIS VCF C6 (2020)")
print("="*70)

comparison_stats = []

for tile, data in results.items():
    if 'modis_vcf_pace' not in data or data['modis_vcf_pace'] is None:
        print(f"\n{tile}: No MODIS VCF available for comparison")
        continue
    
    pace = data['pace_vcf'].flatten()
    modis = data['modis_vcf_pace'].flatten()
    
    valid = np.isfinite(pace) & np.isfinite(modis)
    n_valid = np.sum(valid)
    
    if n_valid < 100:
        print(f"\n{tile}: Insufficient valid pixels ({n_valid})")
        continue
    
    pace_valid = pace[valid]
    modis_valid = modis[valid]
    
    correlation = np.corrcoef(pace_valid, modis_valid)[0, 1]
    r2 = correlation ** 2
    rmse = np.sqrt(np.mean((pace_valid - modis_valid) ** 2))
    mae = np.mean(np.abs(pace_valid - modis_valid))
    bias = np.mean(pace_valid - modis_valid)
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(modis_valid, pace_valid)
    
    stats_dict = {
        'tile': tile,
        'n_pixels': n_valid,
        'correlation': correlation,
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'bias': bias,
        'slope': slope,
        'intercept': intercept,
        'pace_mean': np.mean(pace_valid),
        'pace_std': np.std(pace_valid),
        'modis_mean': np.mean(modis_valid),
        'modis_std': np.std(modis_valid),
    }
    comparison_stats.append(stats_dict)
    
    print(f"\n{tile}:")
    print(f"  Valid pixels:  {n_valid:,}")
    print(f"  Correlation:   {correlation:.4f}")
    print(f"  R²:            {r2:.4f}")
    print(f"  RMSE:          {rmse:.2f}%")
    print(f"  MAE:           {mae:.2f}%")
    print(f"  Bias:          {bias:+.2f}%")
    print(f"  Regression:    y = {slope:.3f}x + {intercept:.2f}")
    print(f"  PACE:          {stats_dict['pace_mean']:.1f}% ± {stats_dict['pace_std']:.1f}%")
    print(f"  MODIS C6:      {stats_dict['modis_mean']:.1f}% ± {stats_dict['modis_std']:.1f}%")

if comparison_stats:
    df_stats = pd.DataFrame(comparison_stats)
    
    print(f"\n{'='*70}")
    print("SUMMARY TABLE - PACE VCF vs MODIS C6 (2020)")
    print(f"{'='*70}")
    print(f"\n{'Tile':<10} {'N Pixels':>10} {'Corr':>8} {'R²':>8} {'RMSE':>8} {'MAE':>8} {'Bias':>8}")
    print("-" * 70)
    for _, row in df_stats.iterrows():
        print(f"{row['tile']:<10} {row['n_pixels']:>10,} {row['correlation']:>8.4f} {row['r2']:>8.4f} {row['rmse']:>8.2f} {row['mae']:>8.2f} {row['bias']:>+8.2f}")
    
    print("-" * 70)
    print(f"{'MEAN':<10} {df_stats['n_pixels'].mean():>10,.0f} {df_stats['correlation'].mean():>8.4f} {df_stats['r2'].mean():>8.4f} {df_stats['rmse'].mean():>8.2f} {df_stats['mae'].mean():>8.2f} {df_stats['bias'].mean():>+8.2f}")
    
    stats_path = OUTPUT_DIR / "comparison_statistics_C6_2020.csv"
    df_stats.to_csv(stats_path, index=False)
    print(f"\nSaved: {stats_path}")


# =============================================================================
# Cell 8: Visualization - Side-by-Side Maps with RGB
# =============================================================================

def load_rgb_from_metrics(tile: str) -> np.ndarray:
    """Load RGB bands from MODIS_Metrics.tif and create a true-color composite."""
    metrics_path = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "MODIS_Metrics.tif"
    
    if not metrics_path.exists():
        print(f"    ⚠️ MODIS_Metrics.tif not found for {tile}")
        return None
    
    rgb_bands = {
        'red': 'BandReflMedian-Band_1',
        'green': 'BandReflMedian-Band_4',
        'blue': 'BandReflMedian-Band_3'
    }
    
    rgb_data = {}
    
    with rasterio.open(metrics_path) as src:
        band_index = {desc: i + 1 for i, desc in enumerate(src.descriptions) if desc}
        
        for color, band_name in rgb_bands.items():
            if band_name in band_index:
                data = src.read(band_index[band_name]).astype(np.float32)
                data = np.where(data == NO_DATA, np.nan, data)
                rgb_data[color] = data
            else:
                print(f"    ⚠️ Band not found: {band_name}")
                return None
    
    red = np.clip(rgb_data['red'] / 2500, 0, 1)
    green = np.clip(rgb_data['green'] / 2500, 0, 1)
    blue = np.clip(rgb_data['blue'] / 2500, 0, 1)
    
    gamma = 0.8
    red = np.power(red, gamma)
    green = np.power(green, gamma)
    blue = np.power(blue, gamma)
    
    rgb = np.dstack([red, green, blue])
    
    invalid_mask = ~(np.isfinite(rgb_data['red']) & np.isfinite(rgb_data['green']) & np.isfinite(rgb_data['blue']))
    rgb[invalid_mask] = [0.85, 0.85, 0.85]
    
    return rgb


vcf_cmap = plt.cm.YlGn.copy()
vcf_cmap.set_bad(color='lightgray')

diff_cmap = plt.cm.RdBu_r.copy()
diff_cmap.set_bad(color='lightgray')

for tile, data in results.items():
    if 'modis_vcf_pace' not in data or data['modis_vcf_pace'] is None:
        continue
    
    rgb_image = load_rgb_from_metrics(tile)
    
    n_cols = 4 if rgb_image is not None else 3
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    
    col_idx = 0
    
    # Panel 1: RGB True Color
    if rgb_image is not None:
        axes[col_idx].imshow(rgb_image)
        axes[col_idx].set_title(f'True Color RGB\n{tile}', fontsize=14, fontweight='bold')
        axes[col_idx].axis('off')
        axes[col_idx].set_aspect('equal')
        col_idx += 1
    
    # Panel 2: PACE VCF
    im1 = axes[col_idx].imshow(data['pace_vcf'], cmap=vcf_cmap, vmin=0, vmax=100)
    axes[col_idx].set_title(f'PACE VCF ({PACE_YEAR})\n{tile}', fontsize=14, fontweight='bold')
    axes[col_idx].axis('off')
    axes[col_idx].set_aspect('equal')
    plt.colorbar(im1, ax=axes[col_idx], shrink=0.8, pad=0.02, label='Tree Cover (%)')
    col_idx += 1
    
    # Panel 3: MODIS VCF C6
    im2 = axes[col_idx].imshow(data['modis_vcf_pace'], cmap=vcf_cmap, vmin=0, vmax=100)
    axes[col_idx].set_title(f'MODIS C6 ({MODIS_VCF_YEAR})\n{tile}', fontsize=14, fontweight='bold')
    axes[col_idx].axis('off')
    axes[col_idx].set_aspect('equal')
    plt.colorbar(im2, ax=axes[col_idx], shrink=0.8, pad=0.02, label='Tree Cover (%)')
    col_idx += 1
    
    # Panel 4: Difference
    diff = data['pace_vcf'] - data['modis_vcf_pace']
    im3 = axes[col_idx].imshow(diff, cmap=diff_cmap, vmin=-30, vmax=30)
    axes[col_idx].set_title(f'Difference (PACE - MODIS)\n{tile}', fontsize=14, fontweight='bold')
    axes[col_idx].axis('off')
    axes[col_idx].set_aspect('equal')
    plt.colorbar(im3, ax=axes[col_idx], shrink=0.8, pad=0.02, label='Difference (%)')
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / f"comparison_map_C6_{tile}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {fig_path.name}")
    plt.show()


# =============================================================================
# Cell 9: Visualization - Scatter Plots
# =============================================================================

tiles_with_modis = [t for t, d in results.items() if 'modis_vcf_pace' in d and d['modis_vcf_pace'] is not None]
n_tiles = len(tiles_with_modis)

if n_tiles > 0:
    n_cols = min(3, n_tiles)
    n_rows = (n_tiles + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5.5*n_rows))
    if n_tiles == 1:
        axes = np.array([axes])
    axes = axes.flatten() if n_tiles > 1 else axes
    
    for idx, tile in enumerate(tiles_with_modis):
        data = results[tile]
        ax = axes[idx] if n_tiles > 1 else axes[0]
        
        pace = data['pace_vcf'].flatten()
        modis = data['modis_vcf_pace'].flatten()
        valid = np.isfinite(pace) & np.isfinite(modis)
        
        h = ax.hist2d(modis[valid], pace[valid], bins=50, 
                      cmap='YlOrRd', norm=mcolors.LogNorm(), cmin=1)
        
        ax.plot([0, 100], [0, 100], 'k--', linewidth=2, label='1:1')
        
        slope, intercept, r_value, _, _ = stats.linregress(modis[valid], pace[valid])
        x_line = np.array([0, 100])
        ax.plot(x_line, slope * x_line + intercept, 'b-', linewidth=2, 
                label=f'Fit: y={slope:.2f}x+{intercept:.1f}')
        
        r2 = r_value ** 2
        rmse = np.sqrt(np.mean((pace[valid] - modis[valid]) ** 2))
        bias = np.mean(pace[valid] - modis[valid])
        ax.text(0.05, 0.95, f'R² = {r2:.3f}\nRMSE = {rmse:.1f}%\nBias = {bias:+.1f}%\nn = {np.sum(valid):,}',
                transform=ax.transAxes, fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_xlabel(f'MODIS C6 {MODIS_VCF_YEAR} (%)', fontsize=11)
        ax.set_ylabel(f'PACE VCF {PACE_YEAR} (%)', fontsize=11)
        ax.set_title(tile, fontsize=12, fontweight='bold')
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.legend(loc='lower right', fontsize=9)
        ax.set_aspect('equal')
        
        plt.colorbar(h[3], ax=ax, label='Count')
    
    for i in range(n_tiles, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / "comparison_scatter_C6_all.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {fig_path.name}")
    plt.show()


# =============================================================================
# Cell 10: Visualization - Distribution Comparison
# =============================================================================

tiles_with_modis = [t for t, d in results.items() if 'modis_vcf_pace' in d and d['modis_vcf_pace'] is not None]
n_tiles = len(tiles_with_modis)

if n_tiles > 0:
    n_cols = min(3, n_tiles)
    n_rows = (n_tiles + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
    if n_tiles == 1:
        axes = np.array([axes])
    axes = axes.flatten() if n_tiles > 1 else axes
    
    for idx, tile in enumerate(tiles_with_modis):
        data = results[tile]
        ax = axes[idx] if n_tiles > 1 else axes[0]
        
        pace = data['pace_vcf'].flatten()
        modis = data['modis_vcf_pace'].flatten()
        
        bins = np.arange(0, 105, 5)
        
        ax.hist(modis[np.isfinite(modis)], bins=bins, alpha=0.6, 
                label=f'MODIS C6 {MODIS_VCF_YEAR}', color='blue', density=True, edgecolor='darkblue')
        ax.hist(pace[np.isfinite(pace)], bins=bins, alpha=0.6, 
                label=f'PACE {PACE_YEAR}', color='green', density=True, edgecolor='darkgreen')
        
        ax.set_xlabel('Tree Cover (%)', fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.set_title(tile, fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.set_xlim(0, 100)
    
    for i in range(n_tiles, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / "comparison_histograms_C6.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved: {fig_path.name}")
    plt.show()


# =============================================================================
# Cell 11: Summary Report
# =============================================================================

report = f"""
================================================================================
PACE-VCF vs MODIS VCF C6 (2020) Comparison Report
================================================================================

Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

MODEL INFORMATION
-----------------
Model: {MODEL_PATH.name}
Features: {len(FEATURE_NAMES)}

DATA SOURCES
------------
PACE VCF Year: {PACE_YEAR}
MODIS VCF C6 Year: {MODIS_VCF_YEAR}
MODIS VCF C6 Source: {MODIS_VCF_C6_DIR}
Test Tiles: {', '.join(TEST_TILES)}
"""

if comparison_stats:
    df_stats = pd.DataFrame(comparison_stats)
    
    report += f"""
================================================================================
RESULTS SUMMARY
================================================================================

Overall Statistics (mean across {len(df_stats)} tiles):
  Correlation: {df_stats['correlation'].mean():.4f}
  R²:          {df_stats['r2'].mean():.4f}
  RMSE:        {df_stats['rmse'].mean():.2f}%
  MAE:         {df_stats['mae'].mean():.2f}%
  Bias:        {df_stats['bias'].mean():+.2f}%

Per-Tile Results:
"""
    for _, row in df_stats.iterrows():
        report += f"""
  {row['tile']}:
    Pixels: {row['n_pixels']:,}
    R²: {row['r2']:.4f}, RMSE: {row['rmse']:.2f}%, Bias: {row['bias']:+.2f}%
    PACE mean: {row['pace_mean']:.1f}%, MODIS mean: {row['modis_mean']:.1f}%
"""

print(report)

report_path = OUTPUT_DIR / "comparison_report_C6_2020.txt"
with open(report_path, 'w') as f:
    f.write(report)
print(f"\nReport saved: {report_path}")


# =============================================================================
# Cell 12: List Output Files
# =============================================================================

print("="*70)
print("OUTPUT FILES")
print("="*70)

for f in sorted(OUTPUT_DIR.glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<50} {size_mb:>8.2f} MB")

print(f"\nTotal files: {len(list(OUTPUT_DIR.glob('*')))}")
print(f"Output directory: {OUTPUT_DIR}")